# CSCI 316 Assignment 2 — TensorFlow/Keras Implementation
## Arabizi Sentiment Analysis via Transfer Learning

This notebook implements the TensorFlow/Keras version of the sentiment classifier.
It must be run in a **fresh Colab session** (no PyTorch imports) to avoid circular import errors.

In [ ]:
!pip install transformers==4.40.0 tf-keras -q

In [ ]:
# ── Cell 1: TF must be imported FIRST ────────────────────────
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import numpy as np
import pandas as pd
import time
import re

from google.colab import drive
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score

drive.mount('/content/drive')
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Data Loading & Exploration\nLoad the raw Arabizi dataset and the source MSA Arabic dataset to understand the data before preprocessing.

In [ ]:
# ── Load raw datasets ────────────────────────────────────────
BASE = '/content/drive/MyDrive/NLP'

# Source domain: mksaad MSA Arabic tweets
src_train = pd.read_csv(f'{BASE}/Arabic Tweets - sentiment/train_Arabic_tweets_positive_negative.tsv',
                         sep='\t', header=None, names=['label', 'text'])
src_test  = pd.read_csv(f'{BASE}/Arabic Tweets - sentiment/test_Arabic_tweets_positive_negative.tsv',
                         sep='\t', header=None, names=['label', 'text'])
src_df = pd.concat([src_train, src_test], ignore_index=True)

# Target domain: Raïdy Arabizi 3-class
tgt_raw = pd.read_csv(f'{BASE}/Arabizi Tweets/3-class-sentiment-arabizi-ds.csv')

print("=== SOURCE: mksaad MSA Arabic ===")
print(f"Shape: {src_df.shape}")
print(f"Labels: {src_df['label'].value_counts().to_dict()}")
print(f"Sample: {src_df['text'].iloc[0][:100]}...")
print()
print("=== TARGET: Raïdy Arabizi ===")
print(f"Shape: {tgt_raw.shape}")
print(f"Columns: {tgt_raw.columns.tolist()}")
print(f"Labels: {tgt_raw['sentiment'].value_counts().to_dict()}")
print(f"Sample: {tgt_raw['tweet'].iloc[0][:100]}...")
print()
print("Source nulls:", src_df.isnull().sum().to_dict())
print("Target nulls:", tgt_raw.isnull().sum().to_dict())

In [ ]:
# ── Data Visualization ───────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Source domain label distribution
src_df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Source Domain (MSA)\nLabel Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Target domain label distribution (raw)
tgt_raw['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=['#e74c3c', '#95a5a6', '#2ecc71'])
axes[1].set_title('Target Domain (Arabizi)\nLabel Distribution')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# Tweet length distribution comparison
src_df['text_len'] = src_df['text'].astype(str).apply(len)
tgt_raw['text_len'] = tgt_raw['tweet'].astype(str).apply(len)
axes[2].hist(src_df['text_len'], bins=50, alpha=0.5, label='MSA (Source)', color='#3498db')
axes[2].hist(tgt_raw['text_len'], bins=50, alpha=0.5, label='Arabizi (Target)', color='#e67e22')
axes[2].set_title('Tweet Length Distribution')
axes[2].set_xlabel('Character Count')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Source avg length: {src_df['text_len'].mean():.0f} chars")
print(f"Target avg length: {tgt_raw['text_len'].mean():.0f} chars")

## 2. Text Preprocessing\nApply domain-specific cleaning:\n- **MSA Arabic**: Remove URLs/mentions/hashtags, strip diacritics, normalize Alef/Ya variants\n- **Arabizi**: Lowercase, map digit-letters to Arabic Unicode (3→ع, 7→ح, 2→ء, etc.), remove punctuation

In [ ]:
# ── MSA Arabic text cleaner ──────────────────────────────────
def clean_msa_arabic(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)  # diacritics
    text = re.sub(r'[أإآٱ]', 'ا', text)  # Alef normalization
    text = re.sub(r'[ىئ]', 'ي', text)     # Ya normalization
    text = re.sub(r'\u0640', '', text)      # tatweel
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)  # keep Arabic + spaces
    return re.sub(r'\s+', ' ', text).strip()

# ── Arabizi text cleaner ─────────────────────────────────────
def clean_arabizi(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    text = text.lower()
    # Map Arabizi digit-letters to Arabic Unicode
    arabizi_map = {
        '3': 'ع', '2': 'ء', '7': 'ح', '5': 'خ',
        '6': 'ط', '8': 'غ', '9': 'ق', '4': 'ذ',
    }
    for digit, letter in arabizi_map.items():
        text = re.sub(rf'(?<=[a-zA-Z\u0600-\u06FF]){digit}|{digit}(?=[a-zA-Z\u0600-\u06FF])',
                      letter, text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

# Apply cleaners
SRC_LABEL_MAP = {'pos': 1, 'neg': 0}
TGT_LABEL_MAP = {'Positive': 2, 'Neutral': 1, 'Negative': 0}

src_df['label'] = src_df['label'].astype(str).str.strip().str.lower()
src_df['clean_text'] = src_df['text'].apply(clean_msa_arabic)
src_df['label_id'] = src_df['label'].map(SRC_LABEL_MAP)
src_df = src_df.dropna(subset=['clean_text', 'label_id'])
src_df = src_df[src_df['clean_text'].str.len() > 5]

tgt_raw = tgt_raw.drop(columns=['highlight'], errors='ignore')
tgt_raw['clean_text'] = tgt_raw['tweet'].apply(clean_arabizi)
tgt_raw['label_id'] = tgt_raw['sentiment'].map(TGT_LABEL_MAP)
tgt_raw = tgt_raw.dropna(subset=['clean_text', 'label_id'])
tgt_raw['label_id'] = tgt_raw['label_id'].astype(int)

print(f"Source after cleaning: {len(src_df)} rows")
print(src_df['label_id'].value_counts())
print(f"\nTarget after cleaning: {len(tgt_raw)} rows")
print(tgt_raw['label_id'].value_counts())

# Show before/after examples
print("\n── MSA Cleaning Examples ──")
for i in [0, 1]:
    print(f"  Before: {src_df['text'].iloc[i][:80]}...")
    print(f"  After:  {src_df['clean_text'].iloc[i][:80]}...")
    print()

print("── Arabizi Cleaning Examples ──")
for i in [0, 1]:
    print(f"  Before: {tgt_raw['tweet'].iloc[i][:80]}...")
    print(f"  After:  {tgt_raw['clean_text'].iloc[i][:80]}...")
    print()

## 3. OOV (Out-of-Vocabulary) Analysis\nMeasure how well mBERT's tokenizer handles both scripts before and after cleaning.

In [ ]:
# ── OOV Rate Analysis ────────────────────────────────────────
ARABIZI_BASE = '/content/drive/MyDrive/NLP/Arabizi Tweets'
tokenizer = AutoTokenizer.from_pretrained(f'{ARABIZI_BASE}/models')

def oov_rate(texts, label=""):
    total_tokens, unk_tokens = 0, 0
    for text in texts[:500]:
        ids = tokenizer.encode(str(text), add_special_tokens=False)
        decoded = tokenizer.convert_ids_to_tokens(ids)
        total_tokens += len(decoded)
        unk_tokens += decoded.count('[UNK]')
    rate = unk_tokens / max(total_tokens, 1) * 100
    print(f"{label}: {unk_tokens}/{total_tokens} OOV tokens ({rate:.2f}%)")
    return rate

oov_msa    = oov_rate(src_df['clean_text'].tolist(), "MSA (cleaned)")
oov_arb    = oov_rate(tgt_raw['tweet'].tolist(),     "Arabizi (raw)")
oov_arb_cl = oov_rate(tgt_raw['clean_text'].tolist(),"Arabizi (cleaned)")

# Visualize OOV reduction
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['MSA (cleaned)', 'Arabizi (raw)', 'Arabizi (cleaned)'],
              [oov_msa, oov_arb, oov_arb_cl],
              color=['#3498db', '#e74c3c', '#2ecc71'])
ax.set_ylabel('OOV Rate (%)')
ax.set_title('Out-of-Vocabulary Token Rates\nmBERT Tokenizer')
for bar, val in zip(bars, [oov_msa, oov_arb, oov_arb_cl]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nArabizi OOV reduction: {oov_arb:.2f}% → {oov_arb_cl:.2f}% ({(1 - oov_arb_cl/oov_arb)*100:.1f}% reduction)")

In [ ]:
# ── Load processed splits from Drive ─────────────────────────
tgt_train_df = pd.read_csv(f'{ARABIZI_BASE}/target/train.csv')
tgt_val_df   = pd.read_csv(f'{ARABIZI_BASE}/target/val.csv')
tgt_test_df  = pd.read_csv(f'{ARABIZI_BASE}/target/test.csv')

print(f"Train: {len(tgt_train_df)} | Val: {len(tgt_val_df)} | Test: {len(tgt_test_df)}")
print("\nLabel distribution per split:")
for name, df in [('Train', tgt_train_df), ('Val', tgt_val_df), ('Test', tgt_test_df)]:
    dist = df['label_id'].value_counts().sort_index().to_dict()
    print(f"  {name}: {dist}")

# Visualize split distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']
colors = ['#e74c3c', '#95a5a6', '#2ecc71']
for ax, (name, df) in zip(axes, [('Train', tgt_train_df), ('Val', tgt_val_df), ('Test', tgt_test_df)]):
    counts = df['label_id'].value_counts().sort_index()
    ax.bar(LABEL_NAMES, counts.values, color=colors)
    ax.set_title(f'{name} Set (n={len(df)})')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.suptitle('Stratified Split Distributions', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nSample cleaned texts:")
print(tgt_train_df[['clean_text', 'label_id']].head(5).to_string())

## 5. TF Data Pipeline\nConvert tokenized text into `tf.data.Dataset` objects with batching and prefetching.\nThis is the TensorFlow equivalent of PyTorch's `DataLoader`.

In [ ]:
# ── Cell 3: Create tf.data.Dataset pipelines ────────────────
def make_tf_dataset(texts, labels, tokenizer, batch=16, shuffle=True):
    # Ensure all texts are strings (handle NaN)
    texts = [str(t) if isinstance(t, str) else "" for t in texts]
    enc = tokenizer(texts, truncation=True, padding=True,
                    max_length=128, return_tensors='np')
    ds = tf.data.Dataset.from_tensor_slices(
        ({'input_ids': enc['input_ids'],
          'attention_mask': enc['attention_mask']},
         np.array(list(labels)))
    )
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

# Drop any rows with NaN in clean_text
tgt_train_df = tgt_train_df.dropna(subset=['clean_text'])
tgt_val_df   = tgt_val_df.dropna(subset=['clean_text'])
tgt_test_df  = tgt_test_df.dropna(subset=['clean_text'])

tf_train_ds = make_tf_dataset(tgt_train_df['clean_text'].tolist(), tgt_train_df['label_id'].tolist(), tokenizer)
tf_val_ds   = make_tf_dataset(tgt_val_df['clean_text'].tolist(),   tgt_val_df['label_id'].tolist(),   tokenizer, shuffle=False)
tf_test_ds  = make_tf_dataset(tgt_test_df['clean_text'].tolist(),  tgt_test_df['label_id'].tolist(),  tokenizer, shuffle=False)

print(f"Train batches: {tf.data.experimental.cardinality(tf_train_ds).numpy()}")
print(f"Val batches:   {tf.data.experimental.cardinality(tf_val_ds).numpy()}")
print(f"Test batches:  {tf.data.experimental.cardinality(tf_test_ds).numpy()}")

## 6. Model Architecture\nLoad `bert-base-multilingual-cased` (mBERT) via HuggingFace's TF API.\nThe model is loaded with `from_pt=True` to convert PyTorch weights to TensorFlow format.\n\n**Key contrast with PyTorch:** TF/Keras uses a declarative `model.compile()` + `model.fit()` approach,\nwhile PyTorch requires an explicit training loop with manual gradient computation.

In [ ]:
# ── Cell 4: Load and compile TF model ───────────────────────
# Load from base mBERT with PyTorch weight conversion
tf_model = TFAutoModelForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
    num_labels=3,
    from_pt=True
)

# Keras declarative compile — contrast with PyTorch imperative loop
tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

tf_model.summary()

## 7. Training\nTrain using Keras `model.fit()` with EarlyStopping callback.\nThis is the declarative counterpart to the PyTorch imperative training loop.

In [ ]:
# ── Train with Keras fit() ───────────────────────────────────
tf_start = time.time()

tf_history = tf_model.fit(
    tf_train_ds,
    validation_data=tf_val_ds,
    epochs=3,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=2,
            restore_best_weights=True,
            monitor='val_loss'
        )
    ]
)

tf_time = time.time() - tf_start
print(f"\nTF training time: {tf_time:.0f}s")

# Save model
tf_model.save_pretrained(f'{ARABIZI_BASE}/models/tensorflow')
print(f"\u2713 TF model saved to {ARABIZI_BASE}/models/tensorflow")

# ── Training History Plots ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(tf_history.history['loss'], 'b-o', label='Train Loss')
axes[0].plot(tf_history.history['val_loss'], 'r-o', label='Val Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(tf_history.history['accuracy'], 'b-o', label='Train Accuracy')
axes[1].plot(tf_history.history['val_accuracy'], 'r-o', label='Val Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('TF/Keras mBERT Training History', fontsize=14)
plt.tight_layout()
plt.savefig(f'{ARABIZI_BASE}/tf_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Print epoch-by-epoch summary
print("\n── Epoch Summary ──")
for i in range(len(tf_history.history['loss'])):
    print(f"  Epoch {i+1}: loss={tf_history.history['loss'][i]:.4f} | "
          f"val_loss={tf_history.history['val_loss'][i]:.4f} | "
          f"acc={tf_history.history['accuracy'][i]:.4f} | "
          f"val_acc={tf_history.history['val_accuracy'][i]:.4f}")

## 8. Test Set Evaluation\nEvaluate the trained TF model on the held-out test set.\nMetrics: classification report (precision/recall/F1 per class), confusion matrix, and per-class analysis.

In [ ]:
# ── Evaluate on test set ─────────────────────────────────────
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']

# Get predictions
tf_preds_logits = tf_model.predict(tf_test_ds)
tf_preds = np.argmax(tf_preds_logits.logits, axis=-1)
tf_true = np.concatenate([y.numpy() for _, y in tf_test_ds])

# Classification report
print("=" * 60)
print("  TF/Keras Full Fine-Tuning (No Transfer)")
print("=" * 60)
print(classification_report(tf_true, tf_preds, target_names=LABEL_NAMES, digits=4))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(tf_true, tf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
test_f1 = f1_score(tf_true, tf_preds, average='macro')
ax.set_title(f'TF/Keras Model\nTest F1: {test_f1:.4f}')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{ARABIZI_BASE}/tf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTest Macro F1: {test_f1:.4f}")
print(f"Training Time: {tf_time:.0f}s")

In [ ]:
# ── Per-class F1 Bar Chart ───────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1_per_class, _ = precision_recall_fscore_support(tf_true, tf_preds, average=None)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(LABEL_NAMES))
width = 0.25
ax.bar(x - width, prec, width, label='Precision', color='#3498db')
ax.bar(x, rec, width, label='Recall', color='#e67e22')
ax.bar(x + width, f1_per_class, width, label='F1-Score', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES)
ax.set_ylabel('Score')
ax.set_title('Per-Class Metrics — TF/Keras Model')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary & Comparison with PyTorch\nCompare the TF/Keras declarative approach with the PyTorch imperative approach used in the main notebook.

In [ ]:
# ── Final Summary ────────────────────────────────────────────
print("=" * 70)
print("  CSCI 316 Assignment 2 — TF/Keras Implementation Summary")
print("=" * 70)

print(f"""
Model:              bert-base-multilingual-cased (mBERT)
Framework:          TensorFlow {tf.__version__} / Keras
Total Parameters:   177,855,747 (678.47 MB)
Training Epochs:    3 (with EarlyStopping, patience=2)
Learning Rate:      2e-5
Batch Size:         16
Max Sequence Len:   128

Dataset:
  Train:  {len(tgt_train_df)} samples
  Val:    {len(tgt_val_df)} samples
  Test:   {len(tgt_test_df)} samples
  Classes: Negative (0), Neutral (1), Positive (2)

Test Results:
  Macro F1:    {test_f1:.4f}
  Train Time:  {tf_time:.0f}s

── PyTorch vs TensorFlow/Keras Comparison ──
┌─────────────────────┬──────────────────┬───────────────────┐
│ Aspect              │ PyTorch          │ TF/Keras          │
├─────────────────────┼──────────────────┼───────────────────┤
│ Training Style      │ Imperative loop  │ Declarative fit() │
│ Gradient Handling   │ Manual backward  │ Automatic         │
│ Data Pipeline       │ DataLoader       │ tf.data.Dataset   │
│ Callbacks           │ Manual logic     │ Built-in system   │
│ Model Saving        │ save_pretrained  │ save_pretrained   │
│ GPU Memory          │ Explicit .to()   │ Automatic         │
└─────────────────────┴──────────────────┴───────────────────┘

Key Insight: Both frameworks achieve comparable results on the same
dataset. TF/Keras requires less boilerplate code for training, while
PyTorch offers more granular control over the training loop.
""")

print("✓ TF/Keras notebook complete")